# 03 — YourMT3+ demo: audio → MIDI, and Strudel → audio → MIDI

This notebook demonstrates both halves of the Restrudel pipeline with the
**pretrained** YourMT3+ model (before any fine-tuning):

1. **WAV → MIDI**: load the YPTF.MoE+Multi (noPS) checkpoint — the variant the
   official demo uses — and transcribe a real electronic track.
2. **Strudel → WAV → MIDI**: render a Strudel REPL snippet to audio *offline in
   Node* (`data_gen/render_offline.mjs`, the Phase 3 spike: SuperDough driven by
   `node-web-audio-api`'s `OfflineAudioContext` — no browser), then feed that
   audio to YourMT3+ and compare the returned MIDI against the pattern's own
   ground-truth events.

The interesting part is where the pretrained model *fails* on synth timbres —
that gap is exactly what the synthetic (audio, MIDI, code) triples from
`data_gen/` are meant to close via fine-tuning.

**Requirements:** repo checkout, Node ≥ 20, `ffmpeg`, and the Python venv from
`requirements.txt` + `torch/torchaudio/torchcodec/lightning/transformers` (see
setup cell). Runs on CPU (~1.5× realtime); CUDA/Colab GPU is faster but optional.


In [1]:
import os, subprocess, sys
from pathlib import Path

REPO = Path.cwd().resolve()
if not (REPO / 'data_gen').exists():          # running from notebooks/
    REPO = REPO.parent
assert (REPO / 'data_gen').exists(), f'repo root not found from {Path.cwd()}'

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Colab ships Python+GPU but needs our deps and Node >= 20
    %pip -q install torch torchaudio torchcodec lightning 'transformers==4.45.1' 'numpy==1.26.4' mido librosa einops deprecated mir_eval huggingface_hub
    !apt-get -qq install -y sox > /dev/null

OUT = REPO / 'notebooks' / 'out'
OUT.mkdir(parents=True, exist_ok=True)
print('repo:', REPO)

repo: /Users/henrik/Documents/Informatik/Master/restrudel


## Setup: fetch YourMT3+ (code + checkpoint) and the offline renderer deps

`scripts/fetch_yourmt3.py` downloads the model code from the official HF Space
(`mimbres/YourMT3`, Apache-2.0) and the ~536 MB **YPTF.MoE+Multi (noPS)**
checkpoint from the HF model repo into `models/YourMT3/` (gitignored). The npm
install pulls Strudel's engine for the renderer; a `postinstall` hook patches
`@kabelsalat/web`'s entry point so `@strudel/core` imports cleanly in Node.


In [2]:
if not (REPO / 'models' / 'YourMT3' / 'amt' / 'src').exists():
    subprocess.run([sys.executable, str(REPO / 'scripts' / 'fetch_yourmt3.py')], check=True)

if not (REPO / 'data_gen' / 'node_modules').exists():
    subprocess.run(['npm', 'install', '--no-fund', '--no-audit'], cwd=REPO / 'data_gen', check=True)

print('model + renderer ready')

model + renderer ready


In [3]:
# Load the model once (CPU; the Space uses the same args with fp16 on CUDA)
sys.path.insert(0, str(REPO / 'scripts'))
from yourmt3_transcribe import load_model, transcribe_file

model = load_model()
print('loaded:', model.__class__.__name__)

/Users/henrik/Documents/Informatik/Master/restrudel/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/henrik/Documents/Informatik/Master/restrudel/models/YourMT3/amt/src/model/RoPE/RoPE.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)
/Users/henrik/Documents/Informatik/Master/restrudel/models/YourMT3/amt/src/model/RoPE/RoPE.py:242: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)


GPU available: True (mps), used: True


TPU available: False, using: 0 TPU cores


/Users/henrik/Documents/Informatik/Master/restrudel/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer(limit_train_batches=1.0)` was configured so 100% of the batches per epoch will be used..


`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..


`Trainer(limit_test_batches=1.0)` was configured so 100% of the batches will be used..


Resuming from amt/logs/2024/mc13_256_g4_all_v7_mt3f_sqr_rms_moe_wf4_n8k2_silu_rope_rp_b36_nops/checkpoints/last.ckpt
Task: mc13_full_plus_256, Max Shift Steps: 206
"add_melody_metric_to_singing": True
"add_pitch_class_metric":       None
"audio_cfg":                    {'codec': 'spec', 'hop_length': 300, 'audio_backend': 'torchaudio', 'sample_rate': 16000, 'input_frames': 32767, 'n_fft': 2048, 'n_mels': 512, 'f_min': 50.0, 'f_max': 8000.0}
"base_lr":                      None
"eval_drum_vocab":              None
"eval_subtask_key":             default
"eval_vocab":                   None
"init_factor":                  None
"max_steps":                    None
"model_cfg":                    {'encoder_type': 'perceiver-tf', 'decoder_type': 'multi-t5', 'pre_encoder_type': 'conv', 'pre_encoder_type_default': {'t5': None, 'perceiver-tf': 'conv', 'conformer': None}, 'pre_decoder_type': 'mc_shared_linear', 'pre_decoder_type_default': {'t5': {'t5': None}, 'perceiver-tf': {'t5': 'linear', 'm

loaded: YourMT3


## Demo 1 — WAV → MIDI on a real track

Clip 20 s out of an electronic track from `mp3s/`, transcribe, and summarize the
MIDI. `transcribe()` reproduces the Space's flow: mono 16 kHz resample →
fixed-length segments → spectrogram (inside the model) → event tokens → notes →
MIDI.


In [4]:
from IPython.display import Audio, display

SRC_MP3 = REPO / 'mp3s' / 'electronic' / 'Vlinderen-144x144-mp4a.mp3'
clip = OUT / 'vlinderen_20s.wav'
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(SRC_MP3),
                '-t', '20', '-ac', '1', str(clip)], check=True)
display(Audio(url=os.path.relpath(clip, REPO / 'notebooks'), embed=False))  # not embedded: keeps the .ipynb small in git

midi_1 = transcribe_file(model, clip, OUT / 'vlinderen_20s.mid')
print('MIDI:', midi_1)

⏰ converting audio: 0m 0s 144.93ms


⏰ model inference: 0m 34s 431.66ms
⏰ post processing: 0m 0s 33.71ms
MIDI: /Users/henrik/Documents/Informatik/Master/restrudel/notebooks/out/vlinderen_20s.mid


In [5]:
import mido
from collections import Counter

def midi_summary(path):
    m = mido.MidiFile(path)
    notes = Counter()
    for tr in m.tracks:
        prog = None
        for msg in tr:
            if msg.type == 'program_change':
                prog = msg.program
            if msg.type == 'note_on' and msg.velocity > 0:
                ch = 'drum' if msg.channel == 9 else f'program {prog}'
                notes[(ch, msg.note)] += 1
    print(f'{path.name}: {m.length:.1f}s, {sum(notes.values())} note-ons')
    for (ch, note), c in sorted(notes.items()):
        print(f'  {ch:12} midi {note:3}  x{c}')
    return notes

_ = midi_summary(midi_1)

vlinderen_20s.mid: 14.1s, 49 note-ons
  program 0    midi  94  x8
  program 0    midi  95  x5
  program 0    midi  97  x11
  program 0    midi  98  x1
  program 65   midi  70  x1
  program 65   midi  71  x8
  program 65   midi  72  x9
  program 65   midi  73  x2
  program 65   midi  74  x4


## Demo 2 — Strudel snippet → WAV → MIDI

A corpus-shaped snippet (sawtooth bass through a low-pass filter + TR-909
drums — the sound distributions `analysis/` found dominant). The renderer
evaluates the same pattern string we would use for label extraction, so audio
and ground truth cannot drift.


In [6]:
STRUDEL_CODE = '''stack(
  note("c2 c2 eb2 g1").s("sawtooth").lpf(800).release(.1),
  s("bd*2, ~ cp, hh*4").bank("RolandTR909")
)'''
CYCLES = 4  # at the default 0.5 cps -> 8s + 2s tail

strudel_wav = OUT / 'strudel_demo.wav'
r = subprocess.run(['node', str(REPO / 'data_gen' / 'render_offline.mjs'),
                    '--code', STRUDEL_CODE, '--cycles', str(CYCLES),
                    '--out', str(strudel_wav)],
                   cwd=REPO / 'data_gen', capture_output=True, text=True)
print(r.stdout[-500:] or r.stderr[-500:])
r.check_returncode()
display(Audio(url=os.path.relpath(strudel_wav, REPO / 'notebooks'), embed=False))

[sampler] load sound "RolandTR909_bd:0:0"..
[sampler] load sound "RolandTR909_bd:0:0"... done! loaded 80.3 KiB in 61ms
[sampler] load sound "RolandTR909_cp:0:0"..
[sampler] load sound "RolandTR909_cp:0:0"... done! loaded 132.8 KiB in 53ms
[sampler] load sound "RolandTR909_hh:0:0"..
[sampler] load sound "RolandTR909_hh:0:0"... done! loaded 20.7 KiB in 24ms
wrote /Users/henrik/Documents/Informatik/Master/restrudel/notebooks/out/strudel_demo.wav: 10.0s, 44100 Hz, 2ch, peak 1.442 (normalized x0.66)



In [7]:
# Ground truth from the pattern itself — the code IS the label (no transcription
# involved): query the same engine for the event list.
import json
gt_script = '''
const { evalScope, noteToMidi } = await import('@strudel/core');
await evalScope(import('@strudel/core'), import('@strudel/mini'));
const { evaluate } = await import('@strudel/transpiler');
const { pattern } = await evaluate(process.argv[1]);
const haps = pattern.queryArc(0, Number(process.argv[2])).filter(h => h.hasOnset());
console.log(JSON.stringify(haps.map(h => ({
  t: h.whole.begin.valueOf() / 0.5,
  s: h.value.s ?? null,
  note: h.value.note != null ? (typeof h.value.note === 'string' ? noteToMidi(h.value.note) : h.value.note) : null,
}))));
'''
r = subprocess.run(['node', '--input-type=module', '-e', gt_script,
                    STRUDEL_CODE, str(CYCLES)],
                   cwd=REPO / 'data_gen', capture_output=True, text=True)
events = json.loads(r.stdout.splitlines()[-1])
gt = Counter((e['s'] or 'note', e['note']) for e in events)
print(f'ground truth: {len(events)} events')
for (s, note), c in sorted(gt.items(), key=str):
    print(f'  {s:10} midi {note}  x{c}')

ground truth: 44 events
  bd         midi None  x8
  cp         midi None  x4
  hh         midi None  x16
  sawtooth   midi 31  x4
  sawtooth   midi 36  x8
  sawtooth   midi 39  x4


In [8]:
midi_2 = transcribe_file(model, strudel_wav, OUT / 'strudel_demo.mid')
pred = midi_summary(midi_2)

⏰ converting audio: 0m 0s 46.17ms


⏰ model inference: 0m 5s 806.73ms
⏰ post processing: 0m 0s 1.74ms
strudel_demo.mid: 8.0s, 27 note-ons
  drum         midi  42  x12
  program 2    midi  31  x4
  program 2    midi  36  x7
  program 2    midi  39  x4


## Reading the result

Typical outcome with the *pretrained* checkpoint on this snippet:

- the **sawtooth bass** (midi 36/39/31) is transcribed almost perfectly — pitch
  content of subtractive synths survives;
- the **hi-hat** is found (drum 42), but the **909 kick and clap are largely
  missed** — synthesized percussion is exactly where models trained on sampled
  acoustic instruments break down (<10 % onset F1 on synth-heavy tracks in the
  YourMT3+ paper's electronic subsets).

That failure mode is the project's thesis in one picture: the model has never
*seen* synth timbres in training. Next step (roadmap Phases 2–5): generate
thousands of aligned (audio, MIDI, Strudel code) triples with
`data_gen/generate.mjs` + this renderer, and fine-tune the checkpoint on them.
